# 03 - Hallazgos de retencion por cohortes

Este notebook interpreta la base [cohortes__retencion_mensual__v1.parquet](../data/processed/cohortes__retencion_mensual__v1.parquet) para traducir la matriz de cohortes en hallazgos defendibles de negocio.

Objetivos del notebook:

- resumir la retencion global en checkpoints clave (`M1`, `M3`, `M6`, `M12`);
- identificar cohortes con mejor y peor desempeno relativo;
- comparar paises con suficiente volumen para obtener una lectura mas estable;
- dejar insumos narrativos para README, dashboard y entrevista tecnica.


## Criterios de lectura

- `M1`, `M3`, `M6` y `M12` representan meses transcurridos desde la primera compra de la cohorte.
- Los meses no observables para cohortes recientes permanecen como faltantes y no deben interpretarse como `0`.
- Los meses observables sin recompra ya fueron densificados en la capa `processed`, por lo que se representan como `0.0%`.
- Para el benchmark por pais se mostraran dos vistas:
  - una vista "core" con paises de mayor volumen;
  - una vista ampliada para detectar senales exploratorias en mercados medianos.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

COHORT_PATH = ROOT_DIR / "data" / "processed" / "cohortes__retencion_mensual__v1.parquet"
cohort_df = pd.read_parquet(COHORT_PATH)

resumen_base = pd.DataFrame(
    {
        "metric": [
            "filas",
            "cohortes_globales",
            "cohortes_por_pais",
            "paises",
            "primer_cohort_month",
            "ultimo_activity_month",
            "max_cohort_index",
        ],
        "valor": [
            len(cohort_df),
            int((cohort_df["cohort_scope"] == "all_countries").sum()),
            int((cohort_df["cohort_scope"] == "primary_country").sum()),
            int(cohort_df.loc[cohort_df["cohort_scope"] == "primary_country", "country"].nunique()),
            str(cohort_df["cohort_month"].min()),
            str(cohort_df["activity_month"].max()),
            int(cohort_df["cohort_index"].max()),
        ],
    }
)

display(resumen_base)
cohort_df.head()


,metric,valor
0,filas,4844
1,cohortes_globales,325
2,cohortes_por_pais,4519
3,paises,41
4,primer_cohort_month,2009-12
5,ultimo_activity_month,2011-12
6,max_cohort_index,24


,cohort_scope,country,cohort_month,activity_month,cohort_index,cohort_size,cohort_multicountry_customers,n_customers_active,n_multicountry_customers_active,retention_rate,n_orders,n_lines,total_quantity,total_revenue_gbp,avg_revenue_per_active_customer_gbp,avg_orders_per_active_customer,avg_lines_per_active_customer
0,all_countries,ALL_COUNTRIES,2009-12,2009-12,0,955,3,955,3,1.0,1512,30272,398660,683504.01,715.711005,1.583246,31.698429
1,all_countries,ALL_COUNTRIES,2009-12,2010-01,1,955,3,337,1,0.35288,569,11901,277892,394723.981,1171.287777,1.688427,35.31454
2,all_countries,ALL_COUNTRIES,2009-12,2010-02,2,955,3,319,1,0.334031,564,11451,247565,295931.572,927.685179,1.768025,35.896552
3,all_countries,ALL_COUNTRIES,2009-12,2010-03,3,955,3,406,2,0.425131,717,14228,322261,378663.12,932.667783,1.76601,35.044335
4,all_countries,ALL_COUNTRIES,2009-12,2010-04,4,955,3,363,2,0.380105,629,12634,172017,305901.29,842.703278,1.732782,34.804408


In [2]:
CHECKPOINTS = [1, 3, 6, 12]


def format_checkpoint_table(df: pd.DataFrame, size_cols: list[str]):
    percentage_cols = [col for col in df.columns if str(col).startswith("m")]
    formatters = {col: "{:.1%}" for col in percentage_cols}
    formatters.update({col: "{:,.0f}" for col in size_cols if col in df.columns})
    return df.style.format(formatters)


def build_global_summary(df: pd.DataFrame) -> pd.DataFrame:
    scope_df = df.loc[df["cohort_scope"] == "all_countries"].copy()
    cohort_size = scope_df.groupby("cohort_month")["cohort_size"].first().rename("cohort_size")
    checkpoint_df = (
        scope_df.loc[scope_df["cohort_index"].isin(CHECKPOINTS)]
        .pivot_table(index="cohort_month", columns="cohort_index", values="retention_rate", aggfunc="first")
        .rename(columns=lambda col: f"m{int(col)}")
    )
    return cohort_size.to_frame().join(checkpoint_df).sort_index()


def build_country_benchmark(
    df: pd.DataFrame,
    min_cohort_size_total: int,
    min_n_cohorts: int,
) -> pd.DataFrame:
    scope_df = df.loc[df["cohort_scope"] == "primary_country"].copy()
    checkpoint_df = (
        scope_df.loc[scope_df["cohort_index"].isin(CHECKPOINTS)]
        .pivot_table(index="country", columns="cohort_index", values="retention_rate", aggfunc="mean")
        .rename(columns=lambda col: f"m{int(col)}")
    )
    cohort_size_total = scope_df.groupby("country")["cohort_size"].sum().rename("cohort_size_total")
    n_cohorts = scope_df.groupby("country")["cohort_month"].nunique().rename("n_cohorts")
    benchmark = checkpoint_df.join([cohort_size_total, n_cohorts])
    benchmark = benchmark.loc[
        (benchmark["cohort_size_total"] >= min_cohort_size_total)
        & (benchmark["n_cohorts"] >= min_n_cohorts)
    ].copy()
    return benchmark.sort_values(["m1", "cohort_size_total"], ascending=[False, False])


def build_period_comparison(global_summary: pd.DataFrame) -> pd.DataFrame:
    split_point = len(global_summary) // 2
    period_map = {
        "cohortes_tempranas": global_summary.iloc[:split_point],
        "cohortes_recientes": global_summary.iloc[split_point:],
    }
    rows = []
    for period_name, period_df in period_map.items():
        row = {"periodo": period_name, "n_cohortes": len(period_df)}
        for checkpoint in ["m1", "m3", "m6", "m12"]:
            if checkpoint in period_df.columns:
                row[checkpoint] = period_df[checkpoint].mean()
                row[f"{checkpoint}_cohortes_disponibles"] = int(period_df[checkpoint].notna().sum())
        rows.append(row)
    return pd.DataFrame(rows).set_index("periodo")


def pct(value) -> str:
    if pd.isna(value):
        return "sin ventana"
    return f"{float(value):.1%}"


## Resumen global de checkpoints

La siguiente tabla resume la retencion por cohorte en cuatro puntos de referencia. Esto permite pasar de la matriz completa a una lectura mas ejecutiva.


In [3]:
global_summary = build_global_summary(cohort_df)
global_average = pd.DataFrame(
    {
        "metrica": ["retencion_promedio_m1", "retencion_promedio_m3", "retencion_promedio_m6", "retencion_promedio_m12"],
        "valor": [
            global_summary["m1"].mean(),
            global_summary["m3"].mean(),
            global_summary["m6"].mean(),
            global_summary["m12"].mean(),
        ],
    }
)

display(format_checkpoint_table(global_summary, size_cols=["cohort_size"]))
display(global_average.style.format({"valor": "{:.1%}"}))


,cohort_size,m1,m3,m6,m12
cohort_month,,,,,
2009-12,955,35.3%,42.5%,37.7%,37.6%
2010-01,383,20.6%,30.5%,25.8%,22.2%
2010-02,374,23.8%,29.1%,19.3%,15.2%
2010-03,443,19.0%,24.2%,24.6%,20.1%
2010-04,294,19.4%,16.3%,27.6%,13.9%
2010-05,254,15.7%,17.3%,21.3%,15.4%
2010-06,270,17.4%,20.4%,12.6%,14.8%
2010-07,186,15.6%,29.6%,11.3%,13.4%
2010-08,162,20.4%,32.1%,9.9%,15.4%


,metrica,valor
0,retencion_promedio_m1,21.2%
1,retencion_promedio_m3,21.6%
2,retencion_promedio_m6,17.8%
3,retencion_promedio_m12,18.2%


## Cohortes destacadas y cohortes debiles

Para evitar comparar cohortes demasiado pequenas, los rankings siguientes filtran por `cohort_size >= 100`. Ademas, el ranking de `M6` solo considera cohortes con al menos seis meses de observacion disponibles.


In [4]:
eligible_global = global_summary.loc[global_summary["cohort_size"] >= 100].copy()
best_m1 = eligible_global.dropna(subset=["m1"]).sort_values("m1", ascending=False).head(8)
worst_m1 = eligible_global.dropna(subset=["m1"]).sort_values("m1", ascending=True).head(8)
best_m6 = eligible_global.dropna(subset=["m6"]).sort_values("m6", ascending=False).head(8)
worst_m6 = eligible_global.dropna(subset=["m6"]).sort_values("m6", ascending=True).head(8)

display(Markdown("### Mejores cohortes por M1"))
display(format_checkpoint_table(best_m1, size_cols=["cohort_size"]))

display(Markdown("### Cohortes mas debiles por M1"))
display(format_checkpoint_table(worst_m1, size_cols=["cohort_size"]))

display(Markdown("### Mejores cohortes maduras por M6"))
display(format_checkpoint_table(best_m6, size_cols=["cohort_size"]))

display(Markdown("### Cohortes maduras mas debiles por M6"))
display(format_checkpoint_table(worst_m6, size_cols=["cohort_size"]))


### Mejores cohortes por M1

,cohort_size,m1,m3,m6,m12
cohort_month,,,,,
2009-12,955,35.3%,42.5%,37.7%,37.6%
2011-10,221,32.1%,,,
2011-08,106,27.4%,26.4%,,
2011-09,189,27.0%,14.8%,,
2010-10,377,25.7%,12.5%,13.0%,19.1%
2011-04,106,25.5%,19.8%,17.9%,
2010-02,374,23.8%,29.1%,19.3%,15.2%
2011-05,111,23.4%,16.2%,26.1%,


### Cohortes mas debiles por M1

,cohort_size,m1,m3,m6,m12
cohort_month,,,,,
2011-11,191,14.1%,,,
2010-07,186,15.6%,29.6%,11.3%,13.4%
2010-05,254,15.7%,17.3%,21.3%,15.4%
2011-02,124,16.1%,18.5%,15.3%,
2010-06,270,17.4%,20.4%,12.6%,14.8%
2010-11,325,17.5%,9.5%,12.9%,25.5%
2011-03,179,18.4%,20.1%,20.7%,
2010-03,443,19.0%,24.2%,24.6%,20.1%


### Mejores cohortes maduras por M6

,cohort_size,m1,m3,m6,m12
cohort_month,,,,,
2009-12,955,35.3%,42.5%,37.7%,37.6%
2010-04,294,19.4%,16.3%,27.6%,13.9%
2011-05,111,23.4%,16.2%,26.1%,
2010-01,383,20.6%,30.5%,25.8%,22.2%
2010-03,443,19.0%,24.2%,24.6%,20.1%
2010-05,254,15.7%,17.3%,21.3%,15.4%
2011-03,179,18.4%,20.1%,20.7%,
2010-02,374,23.8%,29.1%,19.3%,15.2%


### Cohortes maduras mas debiles por M6

,cohort_size,m1,m3,m6,m12
cohort_month,,,,,
2011-06,108,23.1%,26.9%,8.3%,
2010-08,162,20.4%,32.1%,9.9%,15.4%
2010-07,186,15.6%,29.6%,11.3%,13.4%
2010-06,270,17.4%,20.4%,12.6%,14.8%
2010-11,325,17.5%,9.5%,12.9%,25.5%
2010-10,377,25.7%,12.5%,13.0%,19.1%
2010-09,243,22.6%,12.3%,13.6%,21.8%
2011-02,124,16.1%,18.5%,15.3%,


## Evolucion entre cohortes tempranas y recientes

Esta comparacion divide las cohortes globales en dos mitades cronologicas. Es una lectura simple, pero util para detectar si la retencion mejora, se mantiene o se debilita con el tiempo.


In [5]:
period_comparison = build_period_comparison(global_summary)
display(
    period_comparison.style.format(
        {
            "m1": "{:.1%}",
            "m3": "{:.1%}",
            "m6": "{:.1%}",
            "m12": "{:.1%}",
            "n_cohortes": "{:,.0f}",
            "m1_cohortes_disponibles": "{:,.0f}",
            "m3_cohortes_disponibles": "{:,.0f}",
            "m6_cohortes_disponibles": "{:,.0f}",
            "m12_cohortes_disponibles": "{:,.0f}",
        }
    )
)


,n_cohortes,m1,m1_cohortes_disponibles,m3,m3_cohortes_disponibles,m6,m6_cohortes_disponibles,m12,m12_cohortes_disponibles
periodo,,,,,,,,,
cohortes_tempranas,12,21.1%,12,23.0%,12,19.1%,12,19.5%,12
cohortes_recientes,13,21.2%,12,19.9%,10,15.6%,7,2.6%,1


## Benchmark por pais

Se construyen dos vistas complementarias:

- `benchmark_core`: paises con `cohort_size_total >= 500` y al menos `10` cohortes. Sirve para comparaciones mas defendibles.
- `benchmark_ampliado`: paises con `cohort_size_total >= 300` y al menos `10` cohortes. Sirve para detectar senales, con mas cautela.


In [6]:
benchmark_core = build_country_benchmark(cohort_df, min_cohort_size_total=500, min_n_cohorts=10)
benchmark_ampliado = build_country_benchmark(cohort_df, min_cohort_size_total=300, min_n_cohorts=10)

display(Markdown("### Benchmark core"))
display(format_checkpoint_table(benchmark_core, size_cols=["cohort_size_total", "n_cohorts"]))

display(Markdown("### Benchmark ampliado"))
display(format_checkpoint_table(benchmark_ampliado, size_cols=["cohort_size_total", "n_cohorts"]))


### Benchmark core

,m1,m3,m6,m12,cohort_size_total,n_cohorts
country,,,,,,
United Kingdom,21.2%,21.7%,17.5%,17.9%,"90,476",25
France,19.3%,20.0%,18.7%,30.5%,"1,273",25
Germany,18.9%,21.7%,20.6%,19.8%,"1,639",25
Spain,17.1%,26.0%,17.2%,18.2%,570,20


### Benchmark ampliado

,m1,m3,m6,m12,cohort_size_total,n_cohorts
country,,,,,,
Netherlands,38.6%,20.5%,22.7%,4.5%,438,11
Portugal,28.3%,23.1%,33.3%,20.5%,364,15
Belgium,27.5%,15.0%,26.5%,29.2%,429,20
United Kingdom,21.2%,21.7%,17.5%,17.9%,"90,476",25
Sweden,19.4%,16.7%,9.1%,15.0%,325,12
France,19.3%,20.0%,18.7%,30.5%,"1,273",25
Germany,18.9%,21.7%,20.6%,19.8%,"1,639",25
Switzerland,18.5%,14.6%,7.1%,25.0%,328,18
Spain,17.1%,26.0%,17.2%,18.2%,570,20


In [7]:
best_m1_cohort = best_m1.index[0]
best_m6_cohort = best_m6.index[0]
core_best_m1_country = benchmark_core["m1"].idxmax()
core_best_m6_country = benchmark_core["m6"].idxmax()
expanded_best_m1_country = benchmark_ampliado["m1"].idxmax()
expanded_best_m6_country = benchmark_ampliado["m6"].idxmax()

hallazgos_markdown = f"""
## Hallazgos principales

- La retencion global promedio se mueve en torno a `M1 = {pct(global_summary['m1'].mean())}`, `M3 = {pct(global_summary['m3'].mean())}`, `M6 = {pct(global_summary['m6'].mean())}` y `M12 = {pct(global_summary['m12'].mean())}`.
- La cohorte global con mejor `M1` entre cohortes de tamano razonable es **{best_m1_cohort}**, con una retencion de **{pct(best_m1.iloc[0]['m1'])}**.
- La cohorte madura con mejor `M6` es **{best_m6_cohort}**, con una retencion de **{pct(best_m6.iloc[0]['m6'])}**.
- Al comparar cohortes tempranas vs recientes, `M1` se mantiene relativamente estable, pero `M3` y `M6` muestran una senal de debilitamiento en la mitad mas reciente del periodo.
- En la vista `core`, **{core_best_m1_country}** lidera `M1` con **{pct(benchmark_core.loc[core_best_m1_country, 'm1'])}**, mientras que **{core_best_m6_country}** lidera `M6` con **{pct(benchmark_core.loc[core_best_m6_country, 'm6'])}**.
- En la vista ampliada aparecen mercados con mejor retencion temprana que el bloque principal, como **{expanded_best_m1_country}** en `M1` y **{expanded_best_m6_country}** en `M6`, pero deben leerse con cautela porque operan con menos volumen que `United Kingdom`, `Germany` o `France`.

## Lectura de negocio

- La base ya sugiere que **retencion temprana** y **retencion a medio plazo** no siempre premian a los mismos segmentos.
- Esto abre dos lineas utiles para la siguiente fase: estudiar **recurrencia por pais** y cruzar estos hallazgos con **RFM** para entender si los clientes que regresan tambien concentran mayor valor monetario.
"""

display(Markdown(hallazgos_markdown))



## Hallazgos principales

- La retencion global promedio se mueve en torno a `M1 = 21.2%`, `M3 = 21.6%`, `M6 = 17.8%` y `M12 = 18.2%`.
- La cohorte global con mejor `M1` entre cohortes de tamano razonable es **2009-12**, con una retencion de **35.3%**.
- La cohorte madura con mejor `M6` es **2009-12**, con una retencion de **37.7%**.
- Al comparar cohortes tempranas vs recientes, `M1` se mantiene relativamente estable, pero `M3` y `M6` muestran una senal de debilitamiento en la mitad mas reciente del periodo.
- En la vista `core`, **United Kingdom** lidera `M1` con **21.2%**, mientras que **Germany** lidera `M6` con **20.6%**.
- En la vista ampliada aparecen mercados con mejor retencion temprana que el bloque principal, como **Netherlands** en `M1` y **Portugal** en `M6`, pero deben leerse con cautela porque operan con menos volumen que `United Kingdom`, `Germany` o `France`.

## Lectura de negocio

- La base ya sugiere que **retencion temprana** y **retencion a medio plazo** no siempre premian a los mismos segmentos.
- Esto abre dos lineas utiles para la siguiente fase: estudiar **recurrencia por pais** y cruzar estos hallazgos con **RFM** para entender si los clientes que regresan tambien concentran mayor valor monetario.


## Implicaciones para la siguiente fase

Con este notebook ya queda cerrada una lectura inicial de retencion. El siguiente paso natural es construir la capa de `RFM`, de forma que podamos responder no solo *quien vuelve*, sino tambien *quien vuelve con mayor valor*.

Preguntas que este notebook deja listas para profundizar despues:

- que perfiles de cliente sostienen mejor `M3` y `M6`;
- si los paises con mejor retencion temprana tambien presentan mejor valor monetario;
- si las cohortes con peor `M1` esconden clientes de alto valor que compran con menos frecuencia.
